# 116_author_code_artifacts_colab

Author-code mode: build author-like postprocess and impulse-response artifacts
from trained runs.

This notebook is intended for Colab and local Jupyter runs.

In [ ]:
import os
import sys
import shlex
import pathlib
import subprocess

import torch


def _find_project_root() -> pathlib.Path:
    here = pathlib.Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "src").is_dir() and (p / "scripts" / "build_author_postprocess_like.py").is_file() and (p / "scripts" / "build_author_ir_like.py").is_file():
            return p
    cand = pathlib.Path("/content/econml")
    if (cand / "src").is_dir() and (cand / "scripts" / "build_author_postprocess_like.py").is_file() and (cand / "scripts" / "build_author_ir_like.py").is_file():
        return cand
    raise RuntimeError(
        "Project root not found. On Colab, clone repo first, for example:\n"
        "!cd /content && git clone https://github.com/codist-posist/econml.git"
    )


PROJECT_ROOT = _find_project_root()
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("CUDA available:", torch.cuda.is_available())


In [ ]:
# Run configuration
ARTIFACTS_ROOT = str(PROJECT_ROOT / "artifacts")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = "float64"
USE_SELECTED = True
CONS_MODE = "author"

# Post-process length (author default: 10000)
POSTPROCESS_T = 10000

# IR settings (author defaults)
IR_NO_STEPS = 1000
IR_NO_STEPS_DECAY = 400
IR_PRESTEPS = 5
IR_GH_N = 3

POLICIES = [
    "taylor",
    "mod_taylor",
    "discretion",
    "commitment",
    "taylor_zlb",
    "mod_taylor_zlb",
    "discretion_zlb",
    "commitment_zlb",
    "taylor_para",
]

print("DEVICE:", DEVICE)
print("DTYPE:", DTYPE)
print("CONS_MODE:", CONS_MODE)
print("POLICIES:", POLICIES)


In [ ]:
# Build author-like postprocess files policy by policy.
ok_post, skip_post = [], []
for pol in POLICIES:
    cmd = [
        sys.executable,
        "scripts/build_author_postprocess_like.py",
        "--artifacts_root", ARTIFACTS_ROOT,
        "--device", DEVICE,
        "--dtype", DTYPE,
        "--T", str(POSTPROCESS_T),
        "--cons-mode", CONS_MODE,
        "--use_selected" if USE_SELECTED else "--no-use_selected",
        "--policies", pol,
    ]
    print("\nRunning:", " ".join(shlex.quote(x) for x in cmd))
    try:
        subprocess.run(cmd, cwd=str(PROJECT_ROOT), check=True)
        ok_post.append(pol)
    except subprocess.CalledProcessError:
        print(f"[skip] postprocess failed for {pol}")
        skip_post.append(pol)

print("\nPostprocess done.")
print("ok:", ok_post)
print("skip:", skip_post)


In [ ]:
# Build author-like IR files policy by policy.
ok_ir, skip_ir = [], []
for pol in POLICIES:
    cmd = [
        sys.executable,
        "scripts/build_author_ir_like.py",
        "--artifacts-root", str(pathlib.Path(ARTIFACTS_ROOT) / "runs"),
        "--policy", pol,
        "--device", DEVICE,
        "--dtype", DTYPE,
        "--use-selected", str(USE_SELECTED).lower(),
        "--cons-mode", CONS_MODE,
        "--no-steps", str(IR_NO_STEPS),
        "--no-steps-decay", str(IR_NO_STEPS_DECAY),
        "--presteps", str(IR_PRESTEPS),
        "--gh-n", str(IR_GH_N),
        "--clean-out-dir", "true",
    ]
    print("\nRunning:", " ".join(shlex.quote(x) for x in cmd))
    try:
        subprocess.run(cmd, cwd=str(PROJECT_ROOT), check=True)
        ok_ir.append(pol)
    except subprocess.CalledProcessError:
        print(f"[skip] IR export failed for {pol}")
        skip_ir.append(pol)

print("\nAll IR exports done.")
print("ok:", ok_ir)
print("skip:", skip_ir)


In [ ]:
from src.table2_builder import _load_run_dir

print("\nArtifact check:")
for pol in POLICIES:
    try:
        run_dir = pathlib.Path(_load_run_dir(str(pathlib.Path(ARTIFACTS_ROOT) / "runs"), pol, use_selected=USE_SELECTED, required_files=()))
    except Exception:
        print(f"\n[{pol}] run not found")
        continue
    ppp = run_dir / "author_postprocess"
    irs = run_dir / "IRS"
    print(f"\n[{pol}] {run_dir}")
    print(" postprocess:", (ppp / "simulated_definitions.npz").exists(), (ppp / "simulated_definitions_NT.npz").exists(), (ppp / "simulated_definitions_SS.npz").exists())
    print(" ir:", (irs / "IR_definitions.npz").exists(), (irs / "IR_states.npz").exists())
